# Chapter 21 — Scaling Laws and Long Context

Every number in Chapter 21, recomputed from the Hoffmann et al. fit the chapter quotes, plus the
long-context arithmetic. Each cell **asserts** what it checks, so `python tests/run_notebooks.py 21`
fails if the chapter and the code disagree. Exercises 21.1 and 21.2 run on tiny models of this
course's architecture trained for seconds on the bundled offline corpus (a small analogue of the
Tiny Shakespeare runs); Exercise 21.3 needs a real long-context model and is not reproduced.

In [ ]:
import math, sys, os
sys.path.insert(0, os.path.abspath(".."))      # run from notebooks/
sys.path.insert(0, os.path.abspath("."))       # or from the repo root
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)
torch.set_num_threads(min(4, torch.get_num_threads()))
E, A, B_, a, b = 1.69, 406.4, 410.7, 0.34, 0.28                 # Hoffmann et al. 2022
def loss(N, D): return E + A / N ** a + B_ / D ** b
def optimal_20(C):                                                 # C = 6ND with D = 20N
    N = math.sqrt(C / 120); return N, C / (6 * N)

## 21.1 — the worked example: was GPT-3 the right size?

In [ ]:
C = 3.14e23
N_opt, D_opt = optimal_20(C)
print(f"N* = {N_opt / 1e9:.1f} B, D* = {D_opt / 1e9:,.0f} B tokens")
assert round(N_opt / 1e9, 1) == 51.2 and round(D_opt / 1e9) == 1023
as_built, opt, llama = loss(175e9, 300e9), loss(N_opt, D_opt), loss(8e9, 15e12)
print(f"GPT-3 {as_built:.4f}   compute-optimal {opt:.4f}   Llama-3-8B {llama:.4f}")
assert (round(as_built, 4), round(opt, 4), round(llama, 4)) == (2.0023, 1.961, 1.9485)
assert opt < as_built and llama < as_built
assert round(175 / (N_opt / 1e9), 1) == 3.4 and round(D_opt / 300e9, 1) == 3.4
assert round(300 / 175, 1) == 1.7 and round(175 / 8) == 22

## 21.1 — the ridge across budgets, the tokens-per-parameter table, the quiz

In [ ]:
ridge = {1e21: (2.9, 58, 2.335), 1e22: (9.1, 183, 2.146), 1e23: (28.9, 577, 2.012), 1e25: (289, 5774, 1.851)}
for C, (n, d, l) in ridge.items():
    N, D = optimal_20(C)
    print(f"C={C:.0e}  N*={N / 1e9:6.1f}B  D*={D / 1e9:6.0f}B  loss {loss(N, D):.3f}")
    assert round(N / 1e9, 1 if N < 1e11 else 0) == n and round(D / 1e9) == d and round(loss(N, D), 3) == l
assert round(math.sqrt(10), 1) == 3.2                                # 10x compute -> ~3.2x each
for (P, T, want) in ((175e9, 300e9, 1.7), (70e9, 1.4e12, 20), (7e9, 2e12, 286), (8e9, 15e12, 1875), (4e9, 36e12, 9000)):
    assert round(T / P, 1 if want < 10 else 0) == want
assert round(1875 / 20) == 94
N, D = optimal_20(1e23)
assert round(N / 1e9) == 29 and round(D / 1e9, -1) == 580          # quiz: ~29B / ~580B
assert 0.8 < (70e9 * 1.4e12) / (280e9 * 300e9) < 1.25 and 280 / 70 == 4   # ~same compute, 4x smaller
print(f"Chinchilla 6ND = {6 * 70e9 * 1.4e12:.2e}, Gopher 6ND = {6 * 280e9 * 300e9:.2e}")

## The fit's own optimum is not at 20 tokens per parameter

**Resolved (November 2026).** This cell found an error in the chapter, and the chapter text was corrected to match the measurement below. The original finding: The worked example imposes D = 20N and calls the result "compute-optimal"
for the Hoffmann fit; the derivation box says "because α ≈ β, N* ∝ C^0.5". With the quoted
constants α = 0.34 ≠ β = 0.28, minimising L(N, C/6N) exactly gives N* ∝ C^(β/(α+β)) = C^0.45 and
D* ∝ C^0.55, so the ratio is not pinned: ~50 tokens/param at 10²¹ FLOPs, **~87 at GPT-3's budget**
(N* ≈ 24.5 B, D* ≈ 2.1 T, loss 1.954 — lower than the 1.9610 the example reports as optimal), and ~122
at 10²⁵. The 20-tokens-per-parameter rule comes from Hoffmann et al.'s other two estimation
approaches, not from this parametric fit (Besiroglu et al. 2024 discuss the mismatch). The
conclusion — GPT-3 was too big for its data — survives; the numbers labelled "optimal" are the
20-tok/param point, not the fit's minimum.

In [ ]:
G = (a * A / (b * B_)) ** (1 / (a + b))
def argmin(C):
    N = G * (C / 6) ** (b / (a + b)); return N, C / (6 * N)
for C in (1e21, 3.14e23, 1e25):
    N, D = argmin(C)
    grid = min(loss(n, C / (6 * n)) for n in torch.logspace(8, 13, 4001).tolist())
    print(f"C={C:.2e}: argmin N={N / 1e9:7.1f}B D={D / 1e9:8.0f}B  {D / N:5.0f} tok/param  loss {loss(N, D):.4f}"
          f"  (D=20N point: {loss(*optimal_20(C)):.4f})")
    assert abs(loss(N, D) - grid) < 1e-6                         # it really is the minimum
    assert loss(N, D) < loss(*optimal_20(C))
N, D = argmin(3.14e23)
assert round(N / 1e9, 1) == 24.5 and round(D / N) == 87 and round(loss(N, D), 3) == 1.954
assert round(b / (a + b), 2) == 0.45

## 21.2 — position interpolation and NTK-aware base scaling, the numbers

In [ ]:
scale = 4096 / 32768
assert scale == 1 / 8 and 32767 * scale == 4095.875                # "position 32767 -> 4095.9"
base_new = 10000 * 8 ** (128 / 126)
print(f"NTK base for s=8, d=128: {base_new:,.0f}")
assert round(base_new, -2) == 82_700

## 21.2 — sliding-window attention: O(T·W) work, and a receptive field that grows with depth

With `mask = (i - j < W) & (j <= i)` each layer reaches W − 1 tokens back, so L layers reach
L·(W − 1) — the chapter's "n_layer × W", to within one token per layer.

In [ ]:
T, W, L = 64, 4, 5
i, j = torch.arange(T)[:, None], torch.arange(T)[None, :]
mask = ((i - j < W) & (j <= i)).float()
assert mask.sum() == sum(min(t + 1, W) for t in range(T))          # ~T*W entries, not T^2
reach = torch.eye(T)
for _ in range(L):
    reach = (mask @ reach).clamp(max=1)
back = (T - 1) - reach[T - 1].nonzero().min().item()
print(f"{L} layers of window {W}: last token sees {back} tokens back (L*(W-1) = {L * (W - 1)})")
assert back == L * (W - 1)

## Exercise 21.1 — a scaling curve from tiny models

Four widths of this course's model trained for the same number of tokens on the offline corpus.
Loss falls smoothly with N, as the chapter expects; the fitted exponent is printed, not asserted —
the chapter's point is that it will *not* be 0.34 at this scale on this data.

In [ ]:
from tfs.model import Config, Model, build_rope_cache
from tfs.train import _fallback_corpus, get_batch
from tfs.tokenizer import CharTokenizer
text = _fallback_corpus()
ctok = CharTokenizer(text)
data = torch.tensor(ctok.encode(text))
train, val = data[:350_000], data[350_000:]
def train_model(d, steps=150, T=64):
    torch.manual_seed(0)
    m = Model(Config(vocab_size=ctok.vocab_size, d_model=d, n_layer=2, n_head=4, n_kv_head=2,
                     d_head=d // 4, d_ff=2 * d, max_T=T))
    opt = torch.optim.AdamW(m.parameters(), lr=3e-3)
    torch.manual_seed(1)
    for _ in range(steps):
        x = get_batch(train, 16, T)
        _, l, _ = m(x, targets=x)
        opt.zero_grad(); l.backward(); opt.step()
    m.eval()
    with torch.no_grad():
        g = torch.Generator().manual_seed(0)
        ix = torch.randint(len(val) - T - 1, (32,), generator=g)
        xv = torch.stack([val[k:k + T] for k in ix])
        vl = m(xv, targets=xv)[1].item()
    n = sum(p.numel() for name, p in m.named_parameters() if "embed" not in name)
    return m, n, vl
runs = {d: train_model(d) for d in (16, 32, 48, 64)}
for d, (_, n, vl) in runs.items():
    print(f"d_model {d:3d}: {n:7,} non-embedding params  val loss {vl:.3f}")
Ns = torch.tensor([float(r[1]) for r in runs.values()]); Ls = torch.tensor([r[2] for r in runs.values()])
slope = torch.linalg.lstsq(torch.stack([Ns.log(), torch.ones(4)], 1), Ls.log()[:, None]).solution[0].item()
print(f"log-log slope of loss vs N: {slope:.3f}")
assert all(Ls[k + 1] < Ls[k] for k in range(3))                   # bigger is better, smoothly

## Exercise 21.2 — extend the context of a model trained at T = 64

The largest model above, evaluated at 128 and 256 tokens with (a) no change, (b) position
interpolation, (c) NTK-aware base scaling — by swapping the RoPE tables `tfs.model.Model` holds as
buffers. Loss is reported separately on the first 64 positions (the trained range) and the last 64.
YaRN is not implemented here.

The chapter's shape reproduces: (a) degrades on positions past 64 and gets worse with length;
(b) PI is worse than (a) even inside the trained range — the price of compressing every frequency;
(c) NTK keeps the trained range intact and degrades least past it.

In [ ]:
lm = runs[64][0]
cfg = lm.cfg
def set_rope(T, base=10000.0, pos_scale=1.0):
    inv = 1.0 / (base ** (torch.arange(0, cfg.d_head, 2).float() / cfg.d_head))
    f = torch.outer(torch.arange(T).float() * pos_scale, inv)
    e = torch.cat((f, f), -1)
    lm.cos, lm.sin, cfg.max_T = e.cos(), e.sin(), T
@torch.no_grad()
def ev(T, **kw):
    set_rope(T, **kw)
    g = torch.Generator().manual_seed(0)
    ix = torch.randint(len(val) - T - 1, (16,), generator=g)
    xv = torch.stack([val[k:k + T] for k in ix])
    lg, _, _ = lm(xv)
    nll = F.cross_entropy(lg[:, :-1].transpose(1, 2), xv[:, 1:], reduction="none")
    return nll[:, :63].mean().item(), nll[:, -64:].mean().item()
res = {}
for T in (128, 256):
    s = T / 64
    res[T] = {"none": ev(T), "PI": ev(T, pos_scale=1 / s), "NTK": ev(T, base=10000 * s ** (cfg.d_head / (cfg.d_head - 2)))}
    for k, (first, last) in res[T].items():
        print(f"T={T}  {k:4}: loss on positions <64 {first:.3f}   last 64 positions {last:.3f}")
set_rope(64); base_loss = ev(64)[0]
assert torch.allclose(lm.cos, build_rope_cache(64, cfg.d_head)[0])
for T in (128, 256):
    r = res[T]
    assert r["none"][1] > r["none"][0] + 0.05                        # (a) degrades past the trained length
    assert r["PI"][0] > r["none"][0] + 0.1                           # (b) PI hurts the trained range
    assert abs(r["NTK"][0] - r["none"][0]) < 0.15                    # (c) NTK leaves it ~intact ...
    assert r["NTK"][1] < r["none"][1]                                # ... and degrades less past it
assert res[256]["none"][1] > res[128]["none"][1]                     # worse the further you go